

# **Descarga de licitaciones Comunidad de Madrid**

Notebook editado el 07/09/2026

---
## **Set-up**
---


Instalación e importación de librerías:

In [1]:
!pip install pandas sodapy sentence-transformers supabase

In [2]:
from datetime import date, datetime, timedelta
import os
import re
import lxml.etree as ET
from sentence_transformers import SentenceTransformer
from supabase import Client, create_client
import time
import requests

Configuración de la Base de Datos:

In [5]:
from google.colab import userdata
SUPABASE_URL = userdata.get("SUPABASE_URL")
SUPABASE_KEY = userdata.get("SUPABASE_KEY")
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

Carga del modelo de embbedings:

In [6]:
print("Cargando modelo de IA (multilingual-e5-small)...")
encoder = SentenceTransformer("intfloat/multilingual-e5-small", device="cpu")


Cargando modelo de IA (multilingual-e5-small)...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

---
## **Funciones**
---

Definiciones:

In [7]:
URL_ATOM_MADRID = "https://contratos-publicos.comunidad.madrid/feed/licitaciones2"

NS = {
    "atom": "http://www.w3.org/2005/Atom",
    "cac": "urn:dgpe:names:draft:codice:schema:xsd:CommonAggregateComponents-2",
    "cbc": "urn:dgpe:names:draft:codice:schema:xsd:CommonBasicComponents-2",
    "cac-place-ext": "urn:dgpe:names:draft:codice-place-ext:schema:xsd:CommonAggregateComponents-2",
    "cbc-place-ext": "urn:dgpe:names:draft:codice-place-ext:schema:xsd:CommonBasicComponents-2",
}

MAPEO_NUTS = {
    "ES11": "Galicia", "ES12": "Principado de Asturias", "ES13": "Cantabria",
    "ES21": "País Vasco", "ES22": "Comunidad Foral de Navarra", "ES23": "La Rioja",
    "ES24": "Aragón", "ES30": "Comunidad de Madrid", "ES41": "Castilla y León",
    "ES42": "Castilla-La Mancha", "ES43": "Extremadura", "ES51": "Cataluña",
    "ES52": "Comunidad Valenciana", "ES53": "Illes Balears", "ES61": "Andalucía",
    "ES62": "Región de Murcia", "ES63": "Ciudad Autónoma de Ceuta",
    "ES64": "Ciudad Autónoma de Melilla", "ES70": "Canarias", "ES300": "Madrid"
}

In [8]:
def normalizar_organo(texto):
    if not texto:
        return ""
    texto = texto.lower().strip()
    texto = re.sub(r'[áàäâ]', 'a', texto)
    texto = re.sub(r'[éèëê]', 'e', texto)
    texto = re.sub(r'[íìïî]', 'i', texto)
    texto = re.sub(r'[óòöô]', 'o', texto)
    texto = re.sub(r'[úùüû]', 'u', texto)
    texto = re.sub(r'[^a-z0-9\s]', '', texto)
    return re.sub(r'\s+', ' ', texto)

def _texto(el, xpath, ns=NS):
    nodo = el.find(xpath, ns)
    return nodo.text.strip() if nodo is not None and nodo.text else None

---
## **Extracción y sincronización**
---

In [10]:
def sincronizar_licitaciones_madrid():
    hoy = datetime.now().date()
    fecha_inicio_rango = datetime(2026, 9, 1).date()
    fecha_fin_rango = datetime(2026, 9, 7).date()

    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    url_actual = URL_ATOM_MADRID

    entries_totales = []
    max_paginas = 10
    paginas_procesadas = 0

    print(f"Descargando datos del feed ATOM de la Comunidad de Madrid...")

    while url_actual and paginas_procesadas < max_paginas:
        paginas_procesadas += 1
        try:
            resp = requests.get(url_actual, headers=headers, timeout=30)
            if resp.status_code != 200:
                break

            parser = ET.XMLParser(recover=True)
            root = ET.fromstring(resp.content, parser=parser)

            entries = root.findall("atom:entry", NS)
            if not entries:
                entries = root.findall(".//{http://www.w3.org/2005/Atom}entry")
            if not entries:
                break

            entries_totales.extend(entries)

            next_link_el = root.find("atom:link[@rel='next']", NS)
            if next_link_el is None:
                next_link_el = root.find(".//{http://www.w3.org/2005/Atom}link[@rel='next']")

            url_actual = next_link_el.get("href") if next_link_el is not None else None
        except Exception as e:
            print(f"Error descargando feed: {e}")
            break

    print(f"Total de entradas encontradas en el feed: {len(entries_totales)}")

    try:
        existentes_resp = supabase.table("licitaciones").select("enlace, titulo, organo").execute()
        mapa_enlaces = {item["enlace"] for item in existentes_resp.data if "enlace" in item}

        registros_existentes = set()
        for item in existentes_resp.data:
            t = str(item.get("titulo", "")).strip().lower()
            o_base = normalizar_organo(item.get("organo", ""))
            if t or o_base:
                registros_existentes.add((t, o_base))

        print(f"Registros cargados desde Supabase para validación: {len(registros_existentes)}")
    except Exception as e:
        print(f"Error conectando con Supabase: {e}")
        return

    licitaciones_validas = []
    filtrados_caducados = 0
    filtrados_duplicados = 0
    enlaces_ya_procesados_en_sesion = set()
    claves_sesion = set()

    for entry in entries_totales:
        enlace_el = entry.find("atom:link", NS)
        enlace = enlace_el.get("href") if enlace_el is not None else ""
        enlace = enlace.strip()

        txt_updated = _texto(entry, "atom:updated")
        txt_published = _texto(entry, "atom:published")
        txt_fecha = txt_updated or txt_published
        if not txt_fecha:
            continue

        fecha_pub = txt_fecha.split("T")[0]
        try:
            date_pub_obj = datetime.strptime(fecha_pub, "%Y-%m-%d").date()
        except ValueError:
            continue

        if not (fecha_inicio_rango <= date_pub_obj <= fecha_fin_rango):
            continue

        end_date_el = entry.find(".//cac:TenderingProcess/cac:TenderSubmissionDeadlinePeriod/cbc:EndDate", NS)
        fecha_fin_str = "No especificada"
        if end_date_el is not None and end_date_el.text:
            fecha_fin_str = end_date_el.text.strip()[:10]
            try:
                cierre_date = datetime.strptime(fecha_fin_str, "%Y-%m-%d").date()
                if cierre_date < hoy:
                    filtrados_caducados += 1
                    continue
            except ValueError:
                pass

        titulo_str = _texto(entry, "atom:title") or "Sin título"

        organo = "Órgano desconocido"
        rutas_organo = [
            ".//cac-place-ext:LocatedContractingParty//cac:PartyName//cbc:Name",
            ".//cac:ContractingParty//cac:PartyName//cbc:Name",
            ".//cbc:PartyName//cbc:Name"
        ]
        for ruta in rutas_organo:
            organo_el = entry.find(ruta, NS)
            if organo_el is not None and organo_el.text and organo_el.text.strip():
                organo = organo_el.text.strip()
                break

        organo_base = normalizar_organo(organo)

        cpv_codigo = "No especificado"
        cpv_elements = entry.findall(".//cac-place-ext:ContractFolderStatus/cac:ProcurementProject/cac:RequiredCommodityClassification/cbc:ItemClassificationCode", NS)
        if not cpv_elements:
            cpv_elements = entry.findall(".//cbc:ItemClassificationCode", NS)
        if cpv_elements:
            cpv_codigo = ", ".join([el.text.strip() for el in cpv_elements if el.text])

        lugar_ejecucion = "Madrid"
        lugar_el = entry.find(".//cac:ProcurementProject/cac:RealizedLocation/cbc:CountrySubentity", NS)
        if lugar_el is not None and lugar_el.text:
            lugar_ejecucion = lugar_el.text.strip()
        else:
            lugar_el = entry.find(".//cac:ProcurementProject/cac:RealizedLocation/cbc:CountrySubentityCode", NS)
            if lugar_el is not None and lugar_el.text:
                lugar_ejecucion = MAPEO_NUTS.get(lugar_el.text.strip(), lugar_el.text.strip())

        importe = 0.0
        presupuesto_el = entry.find(".//cac:BudgetAmount/cbc:EstimatedOverallContractAmount", NS)
        if presupuesto_el is None:
            presupuesto_el = entry.find(".//cac:BudgetAmount/cbc:TaxExclusiveAmount", NS)
        if presupuesto_el is None:
            presupuesto_el = entry.find(".//cac:BudgetAmount/cbc:TotalAmount", NS)
        if presupuesto_el is not None and presupuesto_el.text:
            try:
                importe = float(presupuesto_el.text.strip().replace(",", "."))
            except Exception:
                pass

        clave_duplicado = (titulo_str.strip().lower(), organo_base)
        if (enlace in mapa_enlaces or
            enlace in enlaces_ya_procesados_en_sesion or
            clave_duplicado in registros_existentes or
            clave_duplicado in claves_sesion):
            filtrados_duplicados += 1
            continue

        enlaces_ya_procesados_en_sesion.add(enlace)
        claves_sesion.add(clave_duplicado)

        texto_completo = f"passage: Título: {titulo_str}. Órgano: {organo}. CPV: {cpv_codigo}. Lugar: {lugar_ejecucion}. Importe: {importe} EUR."
        embedding = encoder.encode(texto_completo).tolist()

        elemento = {
            "titulo": titulo_str.strip(),
            "organo": organo,
            "fecha": fecha_pub,
            "importe": importe,
            "enlace": enlace,
            "texto_completo": texto_completo,
            "embedding": embedding,
            "fecha_fin": fecha_fin_str,
            "lugar_ejecucion": lugar_ejecucion,
            "cpv": cpv_codigo,
            "es_novedad": True,
            "es_actualizada": False,
            "fuente": "Comunidad de Madrid"
        }

        licitaciones_validas.append(elemento)

    print(f"\n--- ESTADÍSTICAS COMUNIDAD DE MADRID ---")
    print(f"Descartados por fecha caducada: {filtrados_caducados}")
    print(f"Duplicados evitados (con órgano normalizado): {filtrados_duplicados}")
    print(f"Licitaciones válidas listas para insertar (del 1 al 7 sept): {len(licitaciones_validas)}\n")

    if licitaciones_validas:
        print("Subiendo licitaciones de la Comunidad de Madrid a Supabase...")
        tamano_lote = 15
        max_intentos = 3

        for i in range(0, len(licitaciones_validas), tamano_lote):
            lote = licitaciones_validas[i:i + tamano_lote]
            num_lote = i // tamano_lote + 1
            exito = False

            for intento in range(1, max_intentos + 1):
                try:
                    supabase.table("licitaciones").upsert(lote, on_conflict="enlace").execute()
                    print(f"  -> Lote Comunidad de Madrid {num_lote} procesado con éxito ({len(lote)} registros).")
                    exito = True
                    break
                except Exception as e:
                    print(f"Intento {intento}/{max_intentos} fallido para lote Madrid {num_lote}: {e}")
                    if intento < max_intentos:
                        time.sleep(2 * intento)
                    else:
                        print(f"Error definitivo al subir lote Madrid {num_lote}.")

        print("¡Sincronización de la Comunidad de Madrid completada con éxito!")
    else:
        print("No hay nuevas licitaciones de la Comunidad de Madrid en el rango de fechas indicado.")

if __name__ == "__main__":
    sincronizar_licitaciones_madrid()

Descargando datos del feed ATOM de la Comunidad de Madrid...
Error descargando feed: HTTPSConnectionPool(host='contratos-publicos.comunidad.madrid', port=443): Read timed out. (read timeout=30)
Total de entradas encontradas en el feed: 150
Registros cargados desde Supabase para validación: 8176

--- ESTADÍSTICAS COMUNIDAD DE MADRID ---
Descartados por fecha caducada: 91
Duplicados evitados (con órgano normalizado): 30
Licitaciones válidas listas para insertar (del 1 al 7 sept): 29

Subiendo licitaciones de la Comunidad de Madrid a Supabase...
Intento 1/3 fallido para lote Madrid 1: {'message': 'canceling statement due to statement timeout', 'code': '57014', 'hint': None, 'details': None}
  -> Lote Comunidad de Madrid 1 procesado con éxito (15 registros).
  -> Lote Comunidad de Madrid 2 procesado con éxito (14 registros).
¡Sincronización de la Comunidad de Madrid completada con éxito!
